In [ ]:
#%pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd 
import seaborn as sns

df = sns.load_dataset("titanic")
df.head(5)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [2]:
print("Checking nulls present in this dataset")
df.isnull().sum()

Checking nulls present in this dataset


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [3]:
df = df[
    ["survived", "pclass", "sex", "age", "fare", "embarked"]
]


df = df.dropna()
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  712 non-null    int64  
 1   pclass    712 non-null    int64  
 2   sex       712 non-null    str    
 3   age       712 non-null    float64
 4   fare      712 non-null    float64
 5   embarked  712 non-null    str    
dtypes: float64(2), int64(2), str(2)
memory usage: 38.9 KB


survived    0
pclass      0
sex         0
age         0
fare        0
embarked    0
dtype: int64

## Turning Categorical features into numerical

In [4]:
df = pd.get_dummies(
    df, columns = ["sex", "embarked"]
)

df.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   survived    712 non-null    int64  
 1   pclass      712 non-null    int64  
 2   age         712 non-null    float64
 3   fare        712 non-null    float64
 4   sex_female  712 non-null    bool   
 5   sex_male    712 non-null    bool   
 6   embarked_C  712 non-null    bool   
 7   embarked_Q  712 non-null    bool   
 8   embarked_S  712 non-null    bool   
dtypes: bool(5), float64(2), int64(2)
memory usage: 31.3 KB


In [9]:
X = df.drop("survived", axis = 1)
y = df["survived"]

print(f"Shape of X is : {X.shape}")
print(f"Shape of y is : {y.shape}")

print(X.head())
print(y.head())

if X.shape[0] == y.shape[0] :
    print("Dataset is good to go . Shapes match , we can continue")


Shape of X is : (712, 8)
Shape of y is : (712,)
   pclass   age     fare  sex_female  sex_male  embarked_C  embarked_Q  \
0       3  22.0   7.2500       False      True       False       False   
1       1  38.0  71.2833        True     False        True       False   
2       3  26.0   7.9250        True     False       False       False   
3       1  35.0  53.1000        True     False       False       False   
4       3  35.0   8.0500       False      True       False       False   

   embarked_S  
0        True  
1       False  
2        True  
3        True  
4        True  
0    0
1    1
2    1
3    1
4    0
Name: survived, dtype: int64
Dataset is good to go . Shapes match , we can continue


## Spliting the dataset and training the decision tree 

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size = 0.3, random_state = 42   
)

print(X_train.shape)
print(y_train.shape)

model = DecisionTreeClassifier(
    max_depth = 4,
    random_state = 42
)

model.fit(X_train,y_train)
predictions = model.predict(X_val)

print(predictions[:10])

## Checking for overfitting 
train_predictions = model.predict(X_train)

print(f"Train: {accuracy_score(y_train, train_predictions)} ")
print(f"Validation : {accuracy_score(y_val, predictions)}")

(498, 8)
(498,)
[1 1 1 1 0 1 1 0 1 0]
Train: 0.857429718875502 
Validation : 0.6962616822429907


# Daignostics : Handling Overfitting 
### Checking for different depth values so our tree isnt overfitting the training set


In [27]:
for depth in [1, 2, 3, 4, 5, 8, 12]:
    model = DecisionTreeClassifier(
        max_depth = depth,
        random_state = 42
    )
    model.fit(X_train,y_train)
    predictions = model.predict(X_val)
    train_predictions = model.predict(X_train)

    print(f"At depth {depth} : Train = {accuracy_score(train_predictions, y_train)}")
    print(f"At depth {depth} : Validation = {accuracy_score(predictions, y_val)}")

At depth 1 : Train = 0.7911646586345381
At depth 1 : Validation = 0.7523364485981309
At depth 2 : Train = 0.8052208835341366
At depth 2 : Validation = 0.7570093457943925
At depth 3 : Train = 0.8232931726907631
At depth 3 : Validation = 0.7149532710280374
At depth 4 : Train = 0.857429718875502
At depth 4 : Validation = 0.6962616822429907
At depth 5 : Train = 0.8815261044176707
At depth 5 : Validation = 0.7570093457943925
At depth 8 : Train = 0.9397590361445783
At depth 8 : Validation = 0.705607476635514
At depth 12 : Train = 0.9759036144578314
At depth 12 : Validation = 0.7242990654205608


### Testing the minimum samples required to split : Found that this alone hanldes the overfitting problem too

In [32]:
for samples in [2, 5, 10, 20, 50, 100]:
    model = DecisionTreeClassifier(
        max_depth = None,
        min_samples_split = samples,
        random_state = 42
    )
    model.fit(X_train, y_train)

    print(f"With sample size {samples}: \ntrain = {accuracy_score(y_train, model.predict(X_train))}")
    print(f"validation = {accuracy_score(y_val, model.predict(X_val))}")

With sample size 2: 
train = 0.9919678714859438
validation = 0.7102803738317757
With sample size 5: 
train = 0.9497991967871486
validation = 0.7102803738317757
With sample size 10: 
train = 0.9196787148594378
validation = 0.7242990654205608
With sample size 20: 
train = 0.893574297188755
validation = 0.7429906542056075
With sample size 50: 
train = 0.8493975903614458
validation = 0.7102803738317757
With sample size 100: 
train = 0.8072289156626506
validation = 0.7336448598130841


## Experimented with variable depth and samples and got 75% accuracy which is 2% better than doing varible sample size alone 

In [35]:
for depth in [1, 2, 3, 4, 5, 8, 12]:
    print(f"Statistics at depth {depth}")
    for sampels in [2, 5, 10, 20, 50, 100]:
        model = DecisionTreeClassifier(
            max_depth = depth,
            min_samples_split = sampels,
            random_state = 42
        )
        model.fit(X_train,y_train)
        predictions = model.predict(X_val)
        train_predictions = model.predict(X_train)

        print(f"With sample size {sampels}: \ntrain = {accuracy_score(y_train, model.predict(X_train))}")
        print(f"validation = {accuracy_score(y_val, model.predict(X_val))}")

Statistics at depth 1
With sample size 2: 
train = 0.7911646586345381
validation = 0.7523364485981309
With sample size 5: 
train = 0.7911646586345381
validation = 0.7523364485981309
With sample size 10: 
train = 0.7911646586345381
validation = 0.7523364485981309
With sample size 20: 
train = 0.7911646586345381
validation = 0.7523364485981309
With sample size 50: 
train = 0.7911646586345381
validation = 0.7523364485981309
With sample size 100: 
train = 0.7911646586345381
validation = 0.7523364485981309
Statistics at depth 2
With sample size 2: 
train = 0.8052208835341366
validation = 0.7570093457943925
With sample size 5: 
train = 0.8052208835341366
validation = 0.7570093457943925
With sample size 10: 
train = 0.8052208835341366
validation = 0.7570093457943925
With sample size 20: 
train = 0.8052208835341366
validation = 0.7570093457943925
With sample size 50: 
train = 0.8052208835341366
validation = 0.7570093457943925
With sample size 100: 
train = 0.8052208835341366
validation = 0.757

# Random Forests

In [38]:
from sklearn.ensemble import RandomForestClassifier

for estimaters in [20, 50, 100]:
    model = RandomForestClassifier(
        n_estimators = estimaters,
        max_depth = 4,
        min_samples_split = 50,
        random_state = 42
    )

    model.fit(X_train, y_train)

    print(f"With {estimaters} trees:")
    print(f"Accuracy of random forest : \nTrain: {accuracy_score(model.predict(X_train),y_train)}")
    print(f"Validation: {accuracy_score(model.predict(X_val),y_val)}")

With 20 trees:
Accuracy of random forest : 
Train: 0.8232931726907631
Validation: 0.7990654205607477
With 50 trees:
Accuracy of random forest : 
Train: 0.8253012048192772
Validation: 0.7710280373831776
With 100 trees:
Accuracy of random forest : 
Train: 0.8253012048192772
Validation: 0.7476635514018691


One important difference from your single Decision Tree:

Random Forest doesn't necessarily get closer to 100% training accuracy as depth increases, because it introduces randomness when building the trees.

In [45]:
for depth in [2, 4, 8, 12, None]:
    model = RandomForestClassifier(
        n_estimators = 100,
        max_depth = depth,
        min_samples_split = 50,
        random_state = 42
    )

    model.fit(X_train, y_train)

    print(f"At {depth} depth:")
    print(f"Accuracy of random forest : \nTrain: {accuracy_score(model.predict(X_train),y_train)}")
    print(f"Validation: {accuracy_score(model.predict(X_val),y_val)}")

At 2 depth:
Accuracy of random forest : 
Train: 0.7971887550200804
Validation: 0.7476635514018691
At 4 depth:
Accuracy of random forest : 
Train: 0.8253012048192772
Validation: 0.7476635514018691
At 8 depth:
Accuracy of random forest : 
Train: 0.8353413654618473
Validation: 0.7850467289719626
At 12 depth:
Accuracy of random forest : 
Train: 0.8293172690763052
Validation: 0.7710280373831776
At None depth:
Accuracy of random forest : 
Train: 0.8293172690763052
Validation: 0.7710280373831776
